# Notebook 02 — Exploratory Analysis

**Milestone:** Phase 4 — Exploratory Data Analysis (Section 10.5)
**Source:** validated processed dataset `data/processed/airfoil_self_noise.csv`
**Figures saved to:** `reports/figures/`

Purpose: visualize the distribution of all six variables, analyze the target, inspect pairwise relationships via scatter plots and a correlation heatmap, and give a dedicated frequency-versus-target analysis. Written observations are in plain language and make no causal claims from correlation.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until the repository root (pyproject.toml + src/airfoil_noise) is found."""
    cur = start.resolve()
    for _ in range(6):
        if (cur / "pyproject.toml").exists() and (cur / "src" / "airfoil_noise").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Repository root could not be located")


REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "airfoil_self_noise.csv"
FIG_DIR = REPO_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_PATH)
TARGET = "scaled_sound_pressure_level_db"
FEATURES = [c for c in df.columns if c != TARGET]

UNITS = {
    "frequency_hz": "Hz",
    "angle_of_attack_deg": "degrees",
    "chord_length_m": "m",
    "free_stream_velocity_m_s": "m/s",
    "suction_side_displacement_thickness_m": "m",
    "scaled_sound_pressure_level_db": "dB",
}

plt.rcParams["figure.dpi"] = 110
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Target column:", TARGET)
print("Features:", FEATURES)

Loaded 1,503 rows x 6 columns
Target column: scaled_sound_pressure_level_db
Features: ['frequency_hz', 'angle_of_attack_deg', 'chord_length_m', 'free_stream_velocity_m_s', 'suction_side_displacement_thickness_m']


## Histograms — all six variables

In [2]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, col in enumerate(df.columns):
    ax = axes[i]
    ax.hist(df[col].dropna(), bins=30, edgecolor="black", color="steelblue")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel(f"{col} ({UNITS[col]})")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.3)
fig.suptitle("Histograms — all six variables", fontsize=14, y=1.02)
fig.tight_layout()
out = FIG_DIR / "histograms_all_six_variables.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
print("Saved:", out)


Saved: D:\Project Aero\airfoil-noise-prediction\reports\figures\histograms_all_six_variables.png


## Box plots — all six variables

In [3]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, col in enumerate(df.columns):
    ax = axes[i]
    ax.boxplot(df[col].dropna(), orientation="vertical", showfliers=True)
    ax.set_title(col.replace("_", " ").title())
    ax.set_ylabel(f"{col} ({UNITS[col]})")
    ax.grid(alpha=0.3, axis="y")
fig.suptitle("Box plots — all six variables", fontsize=14, y=1.02)
fig.tight_layout()
out = FIG_DIR / "boxplots_all_six_variables.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
print("Saved:", out)

Saved: D:\Project Aero\airfoil-noise-prediction\reports\figures\boxplots_all_six_variables.png


## Target distribution analysis

Examine the distribution of the target variable `scaled_sound_pressure_level_db` (dB).

In [4]:
target = df[TARGET]
print("Target" )
print(target.describe().to_string())
print("Skewness:", round(float(target.skew()), 4))
print("Excess kurtosis:", round(float(target.kurt()), 4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(target.dropna(), bins=40, edgecolor="black", color="darkorange")
ax.set_title("Target distribution — scaled sound pressure level (dB)")
ax.set_xlabel(f"{TARGET.replace('_',' ').title()} ({UNITS[TARGET]})")
ax.set_ylabel("Count")
ax.grid(alpha=0.3)
fig.tight_layout()
out = FIG_DIR / "target_distribution.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
print("Saved:", out)


Target
count    1503.000000
mean      124.835943
std         6.898657
min       103.380000
25%       120.191000
50%       125.721000
75%       129.995500
max       140.987000
Skewness: -0.419
Excess kurtosis: -0.3142
Saved: D:\Project Aero\airfoil-noise-prediction\reports\figures\target_distribution.png


## Scatter plots — each input feature vs target

In [5]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6), sharey=True)
for ax, col in zip(axes, FEATURES):
    ax.scatter(df[col], df[TARGET], s=12, alpha=0.6, edgecolor="none")
    ax.set_xlabel(f"{col.replace('_', ' ').title()} ({UNITS[col]})")
    ax.set_title(col.replace("_", " ").title())
    ax.grid(alpha=0.3)
axes[0].set_ylabel(f"{TARGET.replace('_', ' ').title()} ({UNITS[TARGET]})")
fig.suptitle("Each input feature vs target (sound pressure level)", fontsize=14, y=1.02)
fig.tight_layout()
out = FIG_DIR / "scatter_features_vs_target.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
print("Saved:", out)


Saved: D:\Project Aero\airfoil-noise-prediction\reports\figures\scatter_features_vs_target.png


## Correlation heatmap

Pearson correlation matrix for all six variables. Correlation indicates linear association only — it does not imply causation.

In [6]:
corr = df.corr(numeric_only=True)
print(corr.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels([c.replace("_", " ") for c in corr.columns], rotation=45, ha="right")
ax.set_yticklabels([c.replace("_", " ") for c in corr.index])
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", 
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Correlation heatmap — all six variables")
fig.tight_layout()
out = FIG_DIR / "correlation_heatmap.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
print("Saved:", out)


                                       frequency_hz  angle_of_attack_deg  chord_length_m  free_stream_velocity_m_s  suction_side_displacement_thickness_m  scaled_sound_pressure_level_db
frequency_hz                                 1.0000              -0.2728         -0.0037                    0.1337                                -0.2301                         -0.3907
angle_of_attack_deg                         -0.2728               1.0000         -0.5049                    0.0588                                 0.7534                         -0.1561
chord_length_m                              -0.0037              -0.5049          1.0000                    0.0038                                -0.2208                         -0.2362
free_stream_velocity_m_s                     0.1337               0.0588          0.0038                    1.0000                                -0.0040                          0.1251
suction_side_displacement_thickness_m       -0.2301               0.75

Saved: D:\Project Aero\airfoil-noise-prediction\reports\figures\correlation_heatmap.png


## Dedicated analysis — frequency vs target

Scatter plot of `frequency_hz` against the target, plus the correlation and the target mean separated by low/high frequency.

In [7]:
freq = df["frequency_hz"]
target = df[TARGET]
corr_freq = float(freq.corr(target))
print(f"Pearson correlation (frequency_hz vs {TARGET}): {corr_freq:.4f}")

median_freq = freq.median()
low = target[freq <= median_freq]
high = target[freq > median_freq]
print(f"Median frequency: {median_freq} Hz")
print(f"Mean target at low (<=median) frequency: {low.mean():.4f} dB (n={len(low)})")
print(f"Mean target at high (>median) frequency: {high.mean():.4f} dB (n={len(high)})")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(freq, target, s=12, alpha=0.6, edgecolor="none", color="teal")
ax.set_xlabel("Frequency_hz (Hz)")
ax.set_ylabel(f"{TARGET.replace('_', ' ').title()} ({UNITS[TARGET]})")
ax.set_title(f"Frequency vs target (r={corr_freq:.3f})")
ax.axvline(median_freq, color="red", linestyle="--", linewidth=1, label=f"median = {median_freq} Hz")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
out = FIG_DIR / "frequency_vs_target.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
print("Saved:", out)


Pearson correlation (frequency_hz vs scaled_sound_pressure_level_db): -0.3907
Median frequency: 1600.0 Hz
Mean target at low (<=median) frequency: 127.2653 dB (n=767)
Mean target at high (>median) frequency: 122.3042 dB (n=736)
Saved: D:\Project Aero\airfoil-noise-prediction\reports\figures\frequency_vs_target.png


## Written observations

All observations below are descriptive statements about this dataset only. They summarize what the validated data shows; they establish no causal relationship, and no observations were removed or data modified.

1. **Target distribution.** The target ranges from 103.38 dB to 140.99 dB (mean 124.84 dB, std 6.90 dB). It is approximately symmetric with a slight left skew (skewness -0.42) and no extreme tails (excess kurtosis -0.31). Values cluster around 120–130 dB, consistent with the experimental sound-pressure measurements.

2. **Frequency vs target (dedicated analysis).** Frequency has the strongest linear association with the target of any input (Pearson r = -0.3907): higher frequencies tend to pair with lower sound-pressure levels in this data. Splitting at the median frequency (1600 Hz), rows at or below the median average 127.27 dB versus 122.30 dB above — a gap of about 5 dB. Frequency itself is highly right-skewed (median 1600 Hz against a 200–20,000 Hz range), so the association is concentrated at the high-frequency tail. This is a correlation, not a causal claim about frequency.

3. **Other input-target associations.** Suction-side displacement thickness has a moderate negative association (r = -0.3127). Angle of attack (r = -0.1561), chord length (r = -0.2362), and free-stream velocity (r = +0.1251) show only weak linear associations with the target. Weak correlations do not imply the variables are irrelevant — they only indicate little linear association in this dataset.

4. **Associations among inputs.** Angle of attack and suction-side displacement thickness correlate strongly with each other (r = +0.7534), suggesting these two inputs move together in the experiments. Angle of attack and chord length show a moderate negative input-input correlation (r = -0.5049). These input-input relationships are worth noting before modeling because correlated inputs can affect coefficient stability; they are descriptive only and not causal.

5. **Overall integrity.** No missing values, no exact duplicates, all values finite. The validated ranges match the data dictionary extrema exactly: frequency 200–20,000 Hz; angle of attack 0–22.2 deg; chord length 0.0254–0.3048 m; velocity 31.7–71.3 m/s; thickness 0.000400682–0.0584113 m; target 103.38–140.99 dB. Nothing was removed or altered.